In [ ]:
import numpy  as np 

import torch  
import torch.nn as nn    
import h5py 

from torch.utils.data import Dataset, DataLoader

import datetime 




In [ ]:
def load_split(path, split="train"):
    with h5py.File(path, "r") as f:
        x      = f["x"][:]
        grp    = f[split]
        names  = sorted(grp.keys())
        B, N   = len(names), len(x)

        params  = np.zeros((B, 2))
        phi_all = np.zeros((B, N))

        for i, name in enumerate(names):
            run           = grp[name]
            params[i, 0]  = run.attrs["mu"]
            params[i, 1]  = run.attrs["delta"]
            phi_all[i]    = run["phi"][:]

    return x, params, phi_all  # params[:,0]=mu  params[:,1]=delta 

In [3]:

# ============================================================
# Dataset
# ============================================================
class DropletDataset(Dataset):
    def __init__(self, params, phi):
        """
        params : (B, 2)  [mu, delta]
        phi    : (B, N)  analytical solution
        """
        self.params = torch.tensor(params, dtype=torch.float32)
        self.phi    = torch.tensor(phi,    dtype=torch.float32)

    def __len__(self):
        return self.params.shape[0]

    def __getitem__(self, idx):
        return self.params[idx], self.phi[idx]   # (2,), (N,)


# ============================================================
# Load from HDF5  →  DataLoader
# ============================================================
def get_dataloaders(path, batch_size=32, num_workers=0):

    x_ref, train_params, train_phi = load_split(path, "train")
    _,     test_params,  test_phi  = load_split(path, "test")

    train_dataset = DropletDataset(train_params, train_phi)
    test_dataset  = DropletDataset(test_params,  test_phi)

    train_loader = DataLoader(
        train_dataset,
        batch_size  = batch_size,
        shuffle     = True,
 
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size  = batch_size,
        shuffle     = False,
       
    )

    return x_ref, train_loader, test_loader



## Requirment 
1.  We will be working on a system in free space, so the V(x) = 0.
 
2. The predicted solution must satisfy a symmetry constraint (which may need to be enforced, e.g., in the decoder);

3. the solution should satisfy a prescribed (L^2) norm, i.e., |\psi|^2 = N, where N depends on parameters (i.e., it is not an independent parameter). 


In [27]:
class Decoder(nn.Module): 
    
    def __init__(self,  latent_dim, hidden_conv_dims, output_sol_dim):
        super().__init__()
        self.latent_dimension = latent_dim 
        
    
        dims = [latent_dim] + hidden_conv_dims

        layers = []

        for i in range(len(dims) - 1):
            layers.append(
                nn.ConvTranspose1d(
                    dims[i],
                    dims[i + 1],
                    kernel_size=4,
                    stride=2,
                    padding=1
                )
            )
            layers.append(nn.ReLU())

       
        layers.append(
            nn.Conv1d(dims[-1],  output_sol_dim, kernel_size=1, stride = 1 ), 
            #nn.ReLU(),
            #nn.Conv1d(int( dims[-1]//2 ) , output_sol_dim, kernel_size=2, stride = 2 )] 
        )

        self.model = nn.Sequential(*layers)

        print(self.model)
    
    def enforce_even_symmetry(self, y):
        """
        y: (B, C, L)
        """
        y_flip = torch.flip(y, dims=[-1])   # reverse spatial dimension
        return 0.5 * (y + y_flip)
    
        
    def forward(self, x):
        output = self.model(x)
        # enforce the symmetry 
        
        output_symmetry = self.enforce_even_symmetry(output) 
        
         
        return output_symmetry.permute(0, 2, 1).squeeze(-1)
            

In [28]:
class LatentModel(nn.Module):
    
    def __init__(self, input_dim, hidden_dims,
                 latent_feature_dim , n_latent, 
                 activation=nn.ReLU, output_activation=None):
        super().__init__()
        self.n_latent = n_latent
        self.latent_feature_dim = latent_feature_dim 
        self.output_dim = n_latent  * latent_feature_dim 
        layers = []
        
        dims = [input_dim] + list(hidden_dims) + [self.output_dim]
      
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(activation())
        if output_activation is not None:
            layers.append(output_activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x is the (B, number_of_parameters )
        output = self.net(x) 
        
        output = output.view(-1, self.latent_feature_dim, self.n_latent)
        return output 

In [31]:
import logging
import os

# ============================================================
# Logger setup
# ============================================================
def setup_logger(log_path):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    logger = logging.getLogger("trainer")
    logger.setLevel(logging.INFO)

    # file handler
    fh = logging.FileHandler(log_path)
    fh.setLevel(logging.INFO)

    # console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)

    logger.addHandler(fh)
    
    logger.addHandler(ch)

    return logger


# ============================================================
# Train
# ============================================================
def train(models, dataloader, optimizer, criterion, device):
    model1, model2 = models[0], models[1]
    model1.train()
    model2.train()

    total_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model2(model1(x))
       
        loss = criterion(pred, y)
        loss.backward()
        print(loss.item(), "loss-----------")
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


# ============================================================
# Evaluate
# ============================================================
def evaluate(models, dataloader, criterion, device):
    model1, model2 = models[0], models[1]
    model1.eval()
    model2.eval()

    total_loss = 0.0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model2(model1(x))
            loss = criterion(pred, y)
            total_loss += loss.item()

    return total_loss / len(dataloader)


# ============================================================
# Trainer
# ============================================================
def trainer(models, train_loader, test_loader, config, logs, device):


    os.makedirs("./checkpoints", exist_ok=True)
    os.makedirs("./logs",        exist_ok=True)

    logger = setup_logger(f"./logs/{config['log_path']}.log")
    logger.info("=" * 60)
    logger.info("Training started")
    logger.info(f"epochs       : {config['epochs']}")
    logger.info(f"learning_rate: {config['learning_rate']}")
    logger.info(f"weight_decay : {config['weight_decay']}")
    logger.info(f"step_size    : {config['step_size']}")
    logger.info(f"gamma        : {config['gamma']}")
    logger.info(f"epoch_save   : {config['epoch_save']}")
    logger.info("=" * 60)

    optimizer = torch.optim.Adam(
        models.parameters(),
        lr           = config["learning_rate"],
        weight_decay = config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size = config["step_size"],
        gamma     = config["gamma"],
    )
    criterion = nn.MSELoss()

    best_val = float("inf")

    for epoch in range(config["epochs"]):

        # ── train ────────────────────────────────────────────
        train_loss = train(models, train_loader, optimizer, criterion, device)
        scheduler.step()

        # ── logging (every epoch) ────────────────────────────
        logger.info(f"Epoch {epoch:05d} | train: {train_loss:.6f}")

        if logs is not None:
            logs.log({"epoch": epoch, "train_loss": train_loss})

        # ── test + save  (every epoch_save epochs) ───────────
        if epoch % config['epoch_save'] == 0:

            test_loss = evaluate(models, test_loader, criterion, device)

            logger.info(
                f"Epoch {epoch:05d} | test : {test_loss:.6f}  "
                f"[lr={scheduler.get_last_lr()[0]:.2e}]"
            )

            if logs is not None:
                logs.log({"epoch": epoch, "test_loss": test_loss})

            # periodic checkpoint
            torch.save(
                {"model1": models[0].state_dict(),
                 "model2": models[1].state_dict()},
                f"./checkpoints/model_epoch_{epoch:05d}.pt"
            )

            # best model
            if test_loss < best_val:
                best_val = test_loss
                torch.save(
                    {"model1": models[0].state_dict(),
                     "model2": models[1].state_dict()},
                    "./checkpoints/best_model.pt"
                )
                logger.info(
                    f"Epoch {epoch:05d} | best model saved  "
                    f"(test_loss={best_val:.6f})"
                )

    logger.info("=" * 60)
    logger.info(f"Training finished | best test loss: {best_val:.6f}")
    logger.info("=" * 60)

    return models

In [ ]:
d 

datetime.date(2026, 4, 25)

In [32]:

now = datetime.datetime.now()          # 调用时才生成

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


config = {
    
    "eq_name": "Ex1",
    "hidden_dims": [128, 128, 128], 
    "latent_feature_dim":16,
    "n_latent":64,
    
    "hidden_conv_dims": [256, 128, 64, 64, 32, 32],

    
    "activation":nn.ReLU,
    "output_activation": None, 
    
    
    "learning_rate": 1e-3,
    "step_size": 1,
    "gamma":0.99, 
    "epochs" : 5, 
    "epoch_save":1, 
    
         
    "batch_size": 32,
    "epochs": 100,
    "weight_decay": 1e-5,
    "dropout": 0.1,


    # data
    "input_dim": 2,
    "output_sol_dim": 1, 


} 

config["log_path"] = f"./logs/train_{config['eq_name']}_{now:%Y-%m-%d_%H%M%S}.log"



sol_name = f"./data/solution_{config['eq_name']}.h5" 

model = LatentModel(input_dim = config["input_dim"], 
                    hidden_dims= config["hidden_dims"],
                    latent_feature_dim=  config["latent_feature_dim"], 
                    n_latent= config["n_latent"],  
                    activation= config["activation"], 
                    output_activation=config["output_activation"]  )


decoder_model = Decoder(latent_dim = config["latent_feature_dim"],
        hidden_conv_dims = config["hidden_conv_dims"],
        output_sol_dim = config["output_sol_dim"])



# ============================================================
# Data Load 
# ============================================================
x_ref, train_loader, test_loader = get_dataloaders(sol_name, batch_size=32)

# verify
for params_batch, phi_batch in train_loader:
    print("train params :", params_batch.shape)   # (B, 2)
    print("train phi    :", phi_batch.shape)       # (B, N)
    break

for params_batch, phi_batch in test_loader:
    print("test  params :", params_batch.shape)
    print("test  phi    :", phi_batch.shape)
    break 


# ============================================================
# Model Load 
# ============================================================
models = nn.ModuleList([  model  ,  decoder_model])

# ── train ────────────────────────────────────────────────
trainer(
    models       = models,
    train_loader = train_loader,
    test_loader  = test_loader,
    config       = config,
    logs         = None,           # replace with wandb.init() if needed
    device       = device,
)



Using device: cpu
Sequential(
  (0): ConvTranspose1d(16, 256, kernel_size=(4,), stride=(2,), padding=(1,))
  (1): ReLU()
  (2): ConvTranspose1d(256, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (3): ReLU()
  (4): ConvTranspose1d(128, 64, kernel_size=(4,), stride=(2,), padding=(1,))
  (5): ReLU()
  (6): ConvTranspose1d(64, 64, kernel_size=(4,), stride=(2,), padding=(1,))
  (7): ReLU()
  (8): ConvTranspose1d(64, 32, kernel_size=(4,), stride=(2,), padding=(1,))
  (9): ReLU()
  (10): ConvTranspose1d(32, 32, kernel_size=(4,), stride=(2,), padding=(1,))
  (11): ReLU()
  (12): Conv1d(32, 1, kernel_size=(1,), stride=(1,))
)


2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | ============================================================
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | Training started
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | epochs       : 100
2026-04-25 15:24:11 | 

train params : torch.Size([32, 2])
train phi    : torch.Size([32, 4096])
test  params : torch.Size([32, 2])
test  phi    : torch.Size([32, 4096])
0.010271340608596802 loss-----------
0.011805332265794277 loss-----------
0.009781522676348686 loss-----------
0.008459732867777348 loss-----------
0.008678928017616272 loss-----------
0.008084100671112537 loss-----------
0.006352952681481838 loss-----------


2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730
2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730
2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730
2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730
2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730
2026-04-25 15:24:17 | Epoch 00000 | train: 0.008730


0.006409135181456804 loss-----------


...

0.006726729217916727 loss-----------
0.007977546192705631 loss-----------
0.007041906472295523 loss-----------
0.0049748229794204235 loss-----------
0.006425556726753712 loss-----------
0.007042489945888519 loss-----------


2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372
2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372
2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372
2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372
2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372
2026-04-25 15:24:21 | Epoch 00001 | train: 0.006372


0.006080312188714743 loss-----------
0.004707987420260906 loss-----------


2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | test : 0.009181  [lr=9.80e-04]
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)
2026-04-25 15:24:21 | Epoch 00001 | best model saved  (test_loss=0.009181)


KeyboardInterrupt: 

In [ ]:
def main():

    now = datetime.datetime.now()          # 调用时才生成

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")


    config = {
        
        "eq_name": "Ex1",
        "hidden_dims": [128, 128, 128], 
        "latent_feature_dim":16,
        "n_latent":64,
        
        "hidden_conv_dims": [256, 128, 64, 64, 32, 32],

        
        "activation":nn.ReLU,
        "output_activation": None, 
        
        
        "learning_rate": 1e-3,
        "step_size": 1,
        "gamma":0.99, 
        "epochs" : 5, 
        "epoch_save":1, 
        
            
        "batch_size": 32,
        "epochs": 100,
        "weight_decay": 1e-5,
        "dropout": 0.1,


        # data
        "input_dim": 2,
        "output_sol_dim": 1, 


    } 

    config["log_path"] = f"./logs/train_{config['eq_name']}_{now:%Y-%m-%d_%H%M%S}.log"



    sol_name = f"./data/solution_{config['eq_name']}.h5" 

    model = LatentModel(input_dim = config["input_dim"], 
                        hidden_dims= config["hidden_dims"],
                        latent_feature_dim=  config["latent_feature_dim"], 
                        n_latent= config["n_latent"],  
                        activation= config["activation"], 
                        output_activation=config["output_activation"]  )


    decoder_model = Decoder(latent_dim = config["latent_feature_dim"],
            hidden_conv_dims = config["hidden_conv_dims"],
            output_sol_dim = config["output_sol_dim"])



    # ============================================================
    # Data Load 
    # ============================================================
    x_ref, train_loader, test_loader = get_dataloaders(sol_name, batch_size=32)

    # verify
    for params_batch, phi_batch in train_loader:
        print("train params :", params_batch.shape)   # (B, 2)
        print("train phi    :", phi_batch.shape)       # (B, N)
        break

    for params_batch, phi_batch in test_loader:
        print("test  params :", params_batch.shape)
        print("test  phi    :", phi_batch.shape)
        break 


    # ============================================================
    # Model Load 
    # ============================================================
    models = nn.ModuleList([  model  ,  decoder_model])

    # ── train ────────────────────────────────────────────────
    trainer(
        models       = models,
        train_loader = train_loader,
        test_loader  = test_loader,
        config       = config,
        logs         = None,           # replace with wandb.init() if needed
        device       = device,
    )



if __name__ == "__main__":
    main() 

SyntaxError: invalid syntax (<ipython-input-14-cecbc36dce38>, line 40)

config: 


latent_models()